[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/alextfkd/TorchCode/blob/master/solutions/40_linear_regression_solution.ipynb)

# ✅ Solution: Linear Regression

**linear regression** を 3 つの異なるアプローチで実装する — 全部 pure PyTorch で。

データ `X` shape `(N, D)` と target `y` shape `(N,)` が与えられて、weight `w` shape `(D,)` と bias `b` (scalar) を求める：

$$\hat{y} = Xw + b$$


> 💡 **どこで使う**：tabular regression のベースライン。深層学習の出発点でもあり、closed-form / GD / nn.Linear の 3 視点で同じ問題を見ると `loss → grad → update` の感覚が掴める。

<details>
<summary>📖 もっと詳しく</summary>

Closed-form は線形代数で一発、N が大きいと数値的に不安定。GD は線形→ニューラルネットの橋渡し。`nn.Linear + loss.backward()` は実務での書き方。

実務: 解釈性が重要な場面（医療・金融）のベースラインとして残り続けている。

</details>

### Signature
```python
class LinearRegression:
    def closed_form(self, X: Tensor, y: Tensor) -> tuple[Tensor, Tensor]: ...
    def gradient_descent(self, X: Tensor, y: Tensor, lr=0.01, steps=1000) -> tuple[Tensor, Tensor]: ...
    def nn_linear(self, X: Tensor, y: Tensor, lr=0.01, steps=1000) -> tuple[Tensor, Tensor]: ...
```

全 method が `(w, b)` を return — `w` は shape `(D,)`、`b` は shape `()`。

<details>
<summary>📐 3 method の手順</summary>

### Method 1 — Closed-Form (Normal Equation)
X を ones column で augment してから解く：

$$\theta = (X_{aug}^T X_{aug})^{-1} X_{aug}^T y$$

または `torch.linalg.lstsq` / `torch.linalg.solve` を使う。

### Method 2 — Gradient Descent from Scratch
`w` と `b` を zeros で初期化。`steps` 回繰り返す：
```
pred = X @ w + b
error = pred - y
grad_w = (2/N) * X^T @ error
grad_b = (2/N) * error.sum()
w -= lr * grad_w
b -= lr * grad_b
```

### Method 3 — PyTorch nn.Linear
`nn.Linear(D, 1)` を作成、`nn.MSELoss` と optimizer (`torch.optim.SGD` など) を使う。
学習後、layer から `w` と `b` を抽出。

</details>

### Rules
- 入出力は全て **PyTorch tensor**
- numpy や sklearn は **使わない**
- `closed_form` は反復最適化を使わない
- `gradient_descent` は勾配を手計算（`autograd` 使わない）
- `nn_linear` は `torch.nn.Linear` と `loss.backward()` を使う

In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q --force-reinstall --no-deps git+https://github.com/alextfkd/TorchCode.git')
except ImportError:
    pass


In [ ]:
import torch
import torch.nn as nn

In [ ]:
# ✅ SOLUTION

class LinearRegression:
    def closed_form(self, X: torch.Tensor, y: torch.Tensor):
        """Normal equation via augmented matrix."""
        N, D = X.shape
        # Augment X with ones column for bias
        X_aug = torch.cat([X, torch.ones(N, 1)], dim=1)  # (N, D+1)
        # Solve (X^T X) theta = X^T y
        theta = torch.linalg.lstsq(X_aug, y).solution      # (D+1,)
        w = theta[:D]
        b = theta[D]
        return w.detach(), b.detach()

    def gradient_descent(self, X: torch.Tensor, y: torch.Tensor,
                         lr: float = 0.01, steps: int = 1000):
        """Manual gradient computation — no autograd."""
        N, D = X.shape
        w = torch.zeros(D)
        b = torch.tensor(0.0)

        for _ in range(steps):
            pred = X @ w + b          # (N,)
            error = pred - y           # (N,)
            grad_w = (2.0 / N) * (X.T @ error)  # (D,)
            grad_b = (2.0 / N) * error.sum()     # scalar
            w = w - lr * grad_w
            b = b - lr * grad_b

        return w, b

    def nn_linear(self, X: torch.Tensor, y: torch.Tensor,
                  lr: float = 0.01, steps: int = 1000):
        """PyTorch nn.Linear with autograd training loop."""
        N, D = X.shape
        layer = nn.Linear(D, 1)
        optimizer = torch.optim.SGD(layer.parameters(), lr=lr)
        loss_fn = nn.MSELoss()

        for _ in range(steps):
            optimizer.zero_grad()
            pred = layer(X).squeeze(-1)  # (N,)
            loss = loss_fn(pred, y)
            loss.backward()
            optimizer.step()

        w = layer.weight.data.squeeze(0)  # (D,)
        b = layer.bias.data.squeeze(0)    # scalar ()
        return w, b

In [ ]:
# Verify
torch.manual_seed(42)
X = torch.randn(100, 3)
true_w = torch.tensor([2.0, -1.0, 0.5])
y = X @ true_w + 3.0

model = LinearRegression()
for name, method in [("Closed-form", model.closed_form),
                      ("Grad Descent", lambda X, y: model.gradient_descent(X, y, lr=0.05, steps=2000)),
                      ("nn.Linear", lambda X, y: model.nn_linear(X, y, lr=0.05, steps=2000))]:
    w, b = method(X, y)
    print(f"{name:13s}  w={w.tolist()}  b={b.item():.4f}")
print(f"{'True':13s}  w={true_w.tolist()}  b=3.0000")

In [ ]:
# ✅ SUBMIT
from torch_judge import check
check("linear_regression")